# การสร้าง Reply Prompt ตาม Paper EMNLP 2025 (1664)
อิงจากเปเปอร์ `2025.emnlp-main.1664.pdf` ในหัวข้อ **4.1 Prompt Design** โดยนำค่า Dissonance จาก `compare_dissonance.ipynb` มาใช้ใน prompt 3 วิธี:
1. **Base Prompting**: ใส่แค่บทสนทนา (text dialogue) ตามปกติ
2. **Emotion-Enhanced Prompting**: เสริมข้อมูลเรื่อง emotion (valence, arousal) เติมเข้าไปใน prompt
3. **Dissonance-Based Prompting**: ถ้ามี Dissonance ระหว่าง text กับ speech จะระบุเงื่อนไข "emotional dissonance" ในการให้โมเดลตอบกลับ


In [1]:
import pandas as pd
import numpy as np

# โหลดข้อมูล VAD Speech
speech_csv = r"c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\wagner\dialogue_2\emotion_results_wavlm_all.csv"
df_speech = pd.read_csv(speech_csv)

# โหลดข้อมูล VAD Text 
text_csv = r"c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\own_script\dialogue_2\dialogue_2_vad_text.csv"
df_text = pd.read_csv(text_csv)

# ทำการ Scale และคำนวณ Dissonance แบบเดียวกับ compare_dissonance.ipynb
speech = df_speech[df_speech["dialogue_id"] == 2].sort_values("utterance_id").reset_index(drop=True)
text   = df_text.sort_values("utterance_id").reset_index(drop=True)

# Scale ไปช่วง [-1, 1]
speech_aro_s = 2 * np.clip(speech["arousal"].values, 0.0, 1.0) - 1
speech_val_s = 2 * np.clip(speech["valence"].values, 0.0, 1.0) - 1

text_aro_s = 2 * ((np.clip(text["arousal_text"].values, 1.0, 5.0) - 1.0) / 4.0) - 1
text_val_s = 2 * ((np.clip(text["valence_text"].values, 1.0, 5.0) - 1.0) / 4.0) - 1

# คำนวณ Dissonance
delta_arousal = np.abs(speech_aro_s - text_aro_s)
delta_valence = np.abs(speech_val_s - text_val_s)

# Threshold = 0.5 ตาม Paper section 3.1
ARO_THR = 0.5
VAL_THR = 0.5
dissonant_any = (delta_arousal > ARO_THR) | (delta_valence > VAL_THR)

# นำข้อมูลที่จำเป็นมารวมกัน
df = pd.DataFrame({
    "utterance_id": text["utterance_id"],
    "text": text["text"],
    "speech_arousal": speech["arousal"].values,
    "speech_valence": speech["valence"].values,
    "dissonance": dissonant_any
})

df.head()


,utterance_id,text,speech_arousal,speech_valence,dissonance
0,1,Why are you bothering me? What's the problem?,0.577664,0.162604,False
1,2,"Ahh that thing again, can you just stay away f...",0.459763,0.500791,False
2,3,I'm fine! I am very good and doing well at the...,0.867881,0.767458,False
3,4,"Besides, you are the one who seems to be doing...",0.379486,0.537600,False


## Prompt Design Methods
เราจะสร้าง 3 functions แทนแต่ละวิธีใน Paper section 4.1

In [2]:
# สมมติ Therapist คือ Role ผู้ตอบ, Patient คือ Role ของ User
def base_prompting(context):
    prompt = f"""Imagine you are a mental health expert interacting with a patient. Based on the dialogue provided, please apply the most suitable intervention technique and specify its name. Then, continue the dialogue with a concise response.
    
Dialogue:
{context}

Your response must be in the form: [Technique] [Your Response]
"""
    return prompt

def emotion_enhanced_prompting(context, arousal, valence):
    # สามารถจำแนกความรู้สึกเบื้องต้น หรือใส่เป็นตัวเลขตรงๆ ก็ได้ ในเปเปอร์มีใช้เป็น Continuous Emotion Prompts
    prompt = f"""Imagine you are a mental health expert interacting with a patient. Based on the dialogue provided, please apply the most appropriate intervention technique and specify its name. Continue the dialogue with a concise response.
For each patient utterance, emotion predictions from speech modalities are provided.
    
Dialogue:
{context}
Patient (emotion - arousal: {arousal:.2f}, valence: {valence:.2f}): ...

Your response must be in the form: [Technique] [Your Response]
"""
    return prompt

def dissonance_based_prompting(context, has_dissonance):
    # ถ้ามี Dissonance ระบุว่า "(emotional dissonance)"
    dissonance_tag = "Patient (emotional dissonance):" if has_dissonance else "Patient:"
    
    prompt = f"""Imagine you are a mental health expert interacting with a patient. Based on the dialogue provided, please apply the most suitable intervention technique and specify its name.
You have also noticed emotional dissonance—when the patient's vocal tone does not match their spoken text sentiment. Continue the dialogue with a concise response.
    
Dialogue:
{context}
{dissonance_tag} ...

Your response must be in the form: [Technique] [Your Response]
"""
    return prompt


## ตัวอย่างการนำไปใช้ (Generating Replies)
ในบล็อกนี้เราจะจำลองการรับ Utterance ล่าสุดของ Patient แล้วใช้ 3 วิธีเพื่อสร้าง prompt
หลังจากได้ Prompt สามารถนำเอา Prompt ไปใช้กับโมเดลอย่างเช่น OpenAI GPT-4, Llama 3 หรืออื่นๆ ได้

In [3]:
# ดึงประโยคแรกมาเทส
row = df.iloc[0]

dialogue_context = f"Patient: {row['text']}"

print("=== 1. Base Prompting ===")
print(base_prompting(dialogue_context))
print("-" * 50)

print("=== 2. Emotion-Enhanced Prompting ===")
print(emotion_enhanced_prompting(dialogue_context, row['speech_arousal'], row['speech_valence']))
print("-" * 50)

print("=== 3. Dissonance-Based Prompting ===")
print(dissonance_based_prompting(dialogue_context, row['dissonance']))
print("-" * 50)


=== 1. Base Prompting ===
Imagine you are a mental health expert interacting with a patient. Based on the dialogue provided, please apply the most suitable intervention technique and specify its name. Then, continue the dialogue with a concise response.

Dialogue:
Patient: Why are you bothering me? What's the problem?

Your response must be in the form: [Technique] [Your Response]

--------------------------------------------------
=== 2. Emotion-Enhanced Prompting ===
Imagine you are a mental health expert interacting with a patient. Based on the dialogue provided, please apply the most appropriate intervention technique and specify its name. Continue the dialogue with a concise response.
For each patient utterance, emotion predictions from speech modalities are provided.

Dialogue:
Patient: Why are you bothering me? What's the problem?
Patient (emotion - arousal: 0.58, valence: 0.16): ...

Your response must be in the form: [Technique] [Your Response]

-------------------------------

## The Generator Section (Hugging Face API / Local Models)
ในบล็อกนี้เราจะนำข้อมูลข้อความและ Prompts ที่เราสร้างจากฟังก์ชันด้านบน มาป้อนให้กับ LLMs รุ่นอย่าง `meta-llama/Meta-Llama-3-8B-Instruct` หรือที่ใกล้เคียงกันครับ
ด้วยข้อจำกัด GPU อย่าง RTX 4060 (8GB VRAM) เราจะทำงานร่วมกับ Quantization 4-Bit ของ BitsAndBytesConfig ผ่านไลบรารี Transformers 

คุณจะต้องติดตั้ง:
`pip install torch transformers accelerate bitsandbytes`


In [4]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# เลือกใช้ Quantized Model ตามที่สเปคเครื่องพอรับไหวครับ
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# แบบมี quantile

# ตั้งค่า 4-Bit เพื่อลดอัตรากิน VRAM เหลือประมาณ ~5-6 GB 
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {model_id} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
)
print("Model Loaded Successfully!")

Loading meta-llama/Meta-Llama-3-8B-Instruct in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model Loaded Successfully!


### Generate reply dialogue

In [6]:
# --------------------------------------------------------

# # แบบไม่มี quantile

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )


def generate_reply(prompt_text):
    messages = [
        {"role": "system", "content": "You are a helpful and experienced mental health professional. Based on the patient dialogue, provide brief, empathetic, and specific therapeutic interventions as instructed."},
        {"role": "user", "content": prompt_text}
    ]
    
    # รัน Chat Template ควบคู่กับ Special Tokens สำหรับโมเดล
    outputs = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True  # บังคับออกเป็น Dictionary ให้ชัวร์ 100% เลย
    ).to(model.device)
    
    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>") # สำหรับ Llama 3
    ]
    
    # ดึงค่า input_ids หรือ Dictionary กลับเพื่อป้องกัน Error Shape
    input_ids = outputs["input_ids"] if isinstance(outputs, dict) else outputs
    
    # คำสั่ง Generate
    output_ids = model.generate(
        input_ids=input_ids,
        attention_mask=outputs.get("attention_mask", None) if isinstance(outputs, dict) else None,
        max_new_tokens=150,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )
    
    # ตัดส่วนที่เป็น prompt ทิ้ง เอาแค่ output ที่ลามะพิมพ์
    response = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)




In [7]:
# ----------------- TEST ----------------- #
# ลองดึง Prompt 3 รูปแบบ ของ turn แรกมาสร้าง Output
print("============= OUTPUT =============")
print("\n[1. Base Prompting Reply]:\n", generate_reply(base_prompting(dialogue_context)))
print("\n[2. Emotion-Enhanced Prompting Reply]:\n", generate_reply(emotion_enhanced_prompting(dialogue_context, row['speech_arousal'], row['speech_valence'])))
print("\n[3. Dissonance-Based Prompting Reply]:\n", generate_reply(dissonance_based_prompting(dialogue_context, row['dissonance'])))


============= OUTPUT =============


AttributeError: 

In [ ]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)


In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)